In [1]:
!pip install torch_geometric torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-2.9.0+cu126.html

Looking in links: https://data.pyg.org/whl/torch-2.9.0+cu126.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 117.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 22.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 23.3 MB/s eta 0:00:00


In [2]:
import torch
import optuna
import torch.nn as nn
import torch.nn.functional as F
import json

from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINConv, global_add_pool
from sklearn import metrics

In [3]:
class GIN(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, dropout, num_classes=1):
        super().__init__()
        self.dropout = dropout
        self.convs = nn.ModuleList()
        self.batch_norms = nn.ModuleList()

        for i in range(num_layers):
            in_dim = input_dim if i == 0 else hidden_dim
            mlp = nn.Sequential(
                nn.Linear(in_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim),
            )
            self.convs.append(GINConv(mlp))
            self.batch_norms.append(nn.BatchNorm1d(hidden_dim))

        self.lin1 = nn.Linear(hidden_dim, hidden_dim)
        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        h_graph = 0
        for conv, batch_norm in zip(self.convs, self.batch_norms):
            x = F.relu(batch_norm(conv(x, edge_index)))
            x = F.dropout(x, self.dropout, training=self.training)
            h_graph = h_graph + global_add_pool(x, batch)  # sum over layers

        h = F.relu(self.lin1(h_graph))
        h = F.dropout(h, self.dropout, training=self.training)
        return self.classifier(h).view(-1)

In [4]:
train_dataset = torch.load('/kaggle/input/datasets/samuelvarchol/graphs-with-autgrpg-1/train_dataset_baseline.pt',weights_only=False)
val_dataset = torch.load('/kaggle/input/datasets/samuelvarchol/graphs-with-autgrpg-1/val_dataset.pt',weights_only=False)

number_of_features = train_dataset[0].num_node_features

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [5]:
def objective(trial: optuna.Trial):
    global model, optimizer, criterion, scheduler, train_loader, val_loader

    hidden_dim = trial.suggest_categorical('hidden_dim', [64, 128, 256, 512])
    num_layers = trial.suggest_int('num_layers', 2, 6)
    dropout = trial.suggest_float('dropout', 0.0, 0.5)
    lr = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    weight_decay = trial.suggest_float('weight_decay', 1e-6, 1e-3, log=True)
    batch_size = trial.suggest_categorical('batch_size', [64, 128, 256])

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size)

    model = GIN(number_of_features, hidden_dim=hidden_dim, num_layers=num_layers, dropout=dropout).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.BCEWithLogitsLoss()
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=3
    )

    n_epochs = 50
    best_val_acc = 0.0
    patience = 10
    patience_counter = 0

    trial_history = []
    
    for epoch in range(1, n_epochs + 1):
        train_loss = train()
        train_acc, train_f1 = test(train_loader)
        val_acc, val_f1 = test(val_loader)
        
        scheduler.step(val_acc)

        trial_history.append({
            'trial': trial.number,
            'epoch': epoch,
            'train_loss': train_loss,
            'train_acc': train_acc,
            'val_acc': val_acc,
            'lr': optimizer.param_groups[0]['lr']
        })
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            patience_counter = 0
        else:
            patience_counter += 1
        
        trial.report(val_acc, epoch)
        
        if trial.should_prune():
            raise optuna.TrialPruned()
        
        if patience_counter >= patience:
            print(f"Trial {trial.number}: Early stopping at epoch {epoch}")
            break
        
        if epoch % 10 == 0:
            print(f"Trial {trial.number} | Epoch {epoch:02d} | "
                  f"Train Loss: {train_loss:.4f} | "
                  f"Val Acc: {val_acc:.4f}")
    
    return best_val_acc

In [6]:
def train():
    model.train()
    total_loss = 0

    for data in train_loader:
        data = data.to(device)
        out = model(data)
        loss = criterion(out, data.y.float())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * data.num_graphs
        
    return total_loss / len(train_loader.dataset)


@torch.no_grad()
def test(loader):
    model.eval()
    predictions = []
    labels = []

    for data in loader:
        data = data.to(device)
        out = model(data)
        pred = (out > 0).float()
        predictions.append(pred.cpu())
        labels.append(data.y.cpu())

    accuracy = metrics.accuracy_score(torch.cat(labels), torch.cat(predictions))
    f1 = metrics.f1_score(torch.cat(labels), torch.cat(predictions))

    return accuracy, f1

In [7]:
study = optuna.create_study(
    direction="maximize", 
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=10),
    study_name="GIN for partial automorphism extension problem with baseline dataset")

study.optimize(objective, n_trials=100)

trials_df = study.trials_dataframe()
trials_df.to_csv("/kaggle/working/optuna_trials_summary.csv", index=False)

trial = study.best_trial
print(f"  Value (Val Acc): {trial.value:.4f}")
print("\n  Params: ")
for key, value in trial.params.items():
    print(f"    {key}: {value}")

best_params = study.best_params
config_path = f"/kaggle/working/best_config.json"
with open(config_path, "w") as f:
    json.dump(best_params, f, indent=4)

[I 2026-03-14 09:33:35,411] A new study created in memory with name: GIN for partial automorphism extension problem with baseline dataset


Trial 0 | Epoch 10 | Train Loss: 0.5386 | Val Acc: 0.7126
Trial 0 | Epoch 20 | Train Loss: 0.4997 | Val Acc: 0.7389
Trial 0 | Epoch 30 | Train Loss: 0.4714 | Val Acc: 0.7399
Trial 0 | Epoch 40 | Train Loss: 0.4198 | Val Acc: 0.7682


[I 2026-03-14 09:38:29,752] Trial 0 finished with value: 0.7773776395482076 and parameters: {'hidden_dim': 512, 'num_layers': 3, 'dropout': 0.12058105927693846, 'lr': 0.0001930184506094651, 'weight_decay': 1.238078597778804e-06, 'batch_size': 256}. Best is trial 0 with value: 0.7773776395482076.


Trial 0 | Epoch 50 | Train Loss: 0.3775 | Val Acc: 0.7718
Trial 1 | Epoch 10 | Train Loss: 0.5688 | Val Acc: 0.6927
Trial 1 | Epoch 20 | Train Loss: 0.5422 | Val Acc: 0.7147
Trial 1 | Epoch 30 | Train Loss: 0.5249 | Val Acc: 0.7374
Trial 1 | Epoch 40 | Train Loss: 0.5147 | Val Acc: 0.7345


[I 2026-03-14 09:44:00,696] Trial 1 finished with value: 0.755770175151416 and parameters: {'hidden_dim': 64, 'num_layers': 4, 'dropout': 0.20273708231109294, 'lr': 0.0017768136888660687, 'weight_decay': 0.0009915013417580265, 'batch_size': 128}. Best is trial 0 with value: 0.7773776395482076.


...

Trial 95 | Epoch 10 | Train Loss: 0.5341 | Val Acc: 0.7101


[I 2026-03-14 14:10:28,106] Trial 95 pruned. 
[I 2026-03-14 14:11:37,665] Trial 96 pruned. 
[I 2026-03-14 14:12:59,139] Trial 97 pruned. 
[I 2026-03-14 14:14:32,391] Trial 98 pruned. 


Trial 99 | Epoch 10 | Train Loss: 0.5287 | Val Acc: 0.7147


[I 2026-03-14 14:17:14,735] Trial 99 pruned. 


  Value (Val Acc): 0.7864

  Params: 
    hidden_dim: 512
    num_layers: 3
    dropout: 0.0659783242256632
    lr: 0.00013045338154165455
    weight_decay: 2.403767811471815e-06
    batch_size: 128
